# Llama 3.1 8B — Egyptian Civil Code QA (Train + Evaluate)

Fine-tunes **Llama-3.1-8B** (4-bit QLoRA via Unsloth) to answer questions about the **Egyptian Civil Code (Law 131 of 1948)**. Each example gives the model the article text plus a question (formal or Egyptian colloquial) and the model writes a faithful answer in formal Arabic.

**File to upload (Step 2):** `egyptian_civil_qa.json` — the output of `generate_qa.py` (fields: `id`, `question`, `context`, `answer`, `style`).

**Runtime:** Colab → Runtime → Change runtime type → **GPU** (T4 is enough).

### How it is set up for good results
1. **Data cleaning.** Duplicate questions are removed, questions that cannot stand alone ("this article", an article number) are removed, and answers that use words not found in their article (likely hallucinated by the generator) are filtered out, so the model learns to stay faithful to the legal text.
2. **Split by article.** All questions about one article stay on the same side (train / validation / test), so the test score reflects articles the model has never seen.
3. **Train on the answer only.** The loss is computed on the answer tokens, not on the copied article text, with an automatic check that this masking worked.
4. **Validation + best checkpoint.** Validation loss is tracked each epoch and the best epoch is kept.
5. **Arabic-aware metrics.** Exact match, ROUGE and BLEU use Arabic normalisation (the default ROUGE tokenizer scores pure Arabic as 0). Results are also split by formal vs colloquial questions.

> ROUGE, BLEU and semantic similarity measure similarity to the reference answer, not legal correctness. Always read a sample of answers yourself (Step 11 shows examples and the weakest answers).

## Step 1 — Install

In [ ]:
!pip install -q unsloth
!pip install -q rouge_score nltk sentence-transformers

## Step 2 — Upload the dataset

In [ ]:
from google.colab import files
import os
import json

# Upload your QA dataset. The laptop may hide the .json extension.
print("Upload your QA dataset: egyptian_civil_qa.json for the FULL run (egyptian_civil_qa_test.json is only the quick test)")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file was uploaded. Please select your QA JSON file.")

print("Uploaded file(s):")
for name in uploaded:
    print(" -", name)

# Prefer the user's actual filename, but accept the original notebook filename too.
preferred_stems = {
    "egyptian_civil_qa_test",
    "egyptian_civil_qa",
}
matches = [
    name for name in uploaded
    if os.path.splitext(os.path.basename(name))[0].lower() in preferred_stems
]

# If the filename differs, accept a single uploaded JSON file.
if not matches:
    json_files = [
        name for name in uploaded
        if name.lower().endswith((".json", ".jsonl"))
    ]
    if len(json_files) == 1:
        matches = json_files

if len(matches) != 1:
    raise ValueError(
        "Could not identify exactly one QA dataset. Upload only "
        "'egyptian_civil_qa_test.json' (or 'egyptian_civil_qa.json'). "
        f"Detected candidates: {matches or list(uploaded.keys())}"
    )

# Save under a clean, dedicated path so a pre-existing folder named
# /content/egyptian_civil_qa_test.json cannot cause IsADirectoryError.
DATA_DIR = "/content/egyptian_law_input"
os.makedirs(DATA_DIR, exist_ok=True)
DATA_PATH = os.path.join(DATA_DIR, "egyptian_civil_qa.json")

with open(DATA_PATH, "wb") as f:
    f.write(uploaded[matches[0]])

print(f"\nQA dataset saved to: {DATA_PATH}")
print(f"Is a file: {os.path.isfile(DATA_PATH)}")


## Step 3 — Settings

In [ ]:
# Unsloth must be imported before transformers / trl / peft.
from unsloth import FastLanguageModel, is_bfloat16_supported

import json, random, re, shutil
import numpy as np
import pandas as pd
import torch
from datasets import Dataset

# DATA_PATH is set by Step 2 after the uploaded file is saved safely.
DATA_PATH      = globals().get("DATA_PATH", "/content/egyptian_law_input/egyptian_civil_qa.json")
OUTPUT_DIR     = "/content/egyptian_lora_model"
BASE_MODEL     = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"
MAX_SEQ_LENGTH = 2048

# Data
SPLIT_SEED       = 42
TEST_FRACTION    = 0.2    # of articles
VAL_FRACTION     = 0.1    # of the remaining training articles
FAITHFULNESS_MIN = 0.4    # min share of answer words that must appear in the article (0 = keep all)

# Training
LORA_R, LORA_ALPHA = 16, 16
EPOCHS, LR = 3, 2e-4
BATCH_SIZE, GRAD_ACCUM = 2, 4
USE_RESPONSE_ONLY = True  # compute loss on the answer only

# Evaluation
SEMANTIC_THRESHOLD = 0.75

# ---- Arabic text normalisation (used for cleaning and for all metrics) ----
_DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u0640]")
_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def normalize_ar(text):
    t = _DIACRITICS.sub("", str(text))
    t = re.sub("[إأآٱ]", "ا", t)
    t = t.replace("ى", "ي").replace("ة", "ه").translate(_DIGITS)
    t = re.sub(r"[^\w\s]", " ", t)          # punctuation, incl. Arabic comma and question mark
    return " ".join(t.lower().split())

# Strip common attached prefixes (ال، وال، بال، ل، ب، و ...) so a paraphrase like "بالعقد" matches "العقد".
_PREFIXES = ("وال", "بال", "كال", "فال", "لل", "ال", "و", "ب", "ل", "ف", "ك")
def strip_prefix(w):
    for p in _PREFIXES:
        if w.startswith(p) and len(w) - len(p) >= 3:
            return w[len(p):]
    return w

## Step 4 — Load, clean, and split **by article**

In [ ]:
with open(DATA_PATH, encoding="utf-8") as f:
    raw = json.load(f)

if not isinstance(raw, list):
    raise TypeError(
        f"Expected the QA JSON file to contain a list of records, got {type(raw).__name__}. "
        "Each record should include question, answer, and context fields."
    )
if raw and not isinstance(raw[0], dict):
    raise TypeError("Expected each QA dataset item to be a JSON object/dictionary.")

import re
_DIAC = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u0640]")
def _norm_q(t):
    t = _DIAC.sub("", str(t))
    t = re.sub("[إأآٱ]", "ا", t).replace("ى", "ي").replace("ة", "ه")
    t = t.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    t = re.sub(r"[^\w\s]", " ", t)
    return " ".join(t.split())

# Questions that cannot be understood without seeing the article ("this article", an article number...).
SELF_REF = re.compile(
    r"(هذه الماده|هذا النص|هذه الفقره|نص الماده|الماده\s*(?:رقم\s*)?\d+"
    r"|(?:وفقا|حسب|بحسب|طبقا)\s+(?:ل)?(?:النص|الماده))")
def is_self_referential(question):
    return bool(SELF_REF.search(_norm_q(question)))

def faithfulness(answer, context):
    # Share of answer words that also appear in the article text.
    a = [strip_prefix(w) for w in normalize_ar(answer).split()]
    c = {strip_prefix(w) for w in normalize_ar(context).split()}
    return sum(t in c for t in a) / max(len(a), 1)

rows, seen, dropped_dup, dropped_unfaithful, dropped_selfref, examples_dropped = [], set(), 0, 0, 0, []
for r in raw:
    q = str(r.get("question", "")).strip()
    a = str(r.get("answer", "")).strip()
    c = str(r.get("context", "")).strip()
    if not (q and a and c):
        continue
    if is_self_referential(q):
        dropped_selfref += 1
        continue
    key = (c, normalize_ar(q))
    if key in seen:
        dropped_dup += 1
        continue
    seen.add(key)
    if faithfulness(a, c) < FAITHFULNESS_MIN:
        dropped_unfaithful += 1
        if len(examples_dropped) < 3:
            examples_dropped.append((q, a))
        continue
    rows.append({"question": q, "answer": a, "context": c,
                 "style": str(r.get("style", "formal")).strip() or "formal"})

dataset = Dataset.from_list(rows)
print(f"Raw pairs               : {len(raw)}")
print(f"Dropped (self-referential): {dropped_selfref}   <- questions like \"according to this article\"")
print(f"Dropped (duplicates)    : {dropped_dup}")
print(f"Dropped (unfaithful)    : {dropped_unfaithful}   <- raise/lower FAITHFULNESS_MIN if this looks too high/low")
print(f"Kept                    : {len(dataset)}  ({len(set(dataset['context']))} articles)")
print("\nQuestion styles:")
print(pd.Series(dataset["style"]).value_counts().to_string())

if examples_dropped:
    print("\nExamples that were dropped as unfaithful:")
    for q, a in examples_dropped:
        print(" Q:", q); print(" A:", a[:160]); print()

In [ ]:
contexts = sorted(set(dataset["context"]))
random.Random(SPLIT_SEED).shuffle(contexts)

n_test = int(len(contexts) * TEST_FRACTION)
test_contexts = set(contexts[:n_test])
rest = contexts[n_test:]
n_val = int(len(rest) * VAL_FRACTION)
val_contexts = set(rest[:n_val])

train_dataset = dataset.filter(lambda ex: ex["context"] not in test_contexts and ex["context"] not in val_contexts)
val_dataset   = dataset.filter(lambda ex: ex["context"] in val_contexts)
test_dataset  = dataset.filter(lambda ex: ex["context"] in test_contexts)

# Hard guarantee: no article appears in two splits.
tr_c, va_c, te_c = (set(d["context"]) for d in (train_dataset, val_dataset, test_dataset))
assert not (tr_c & va_c) and not (tr_c & te_c) and not (va_c & te_c), "Leakage: shared articles!"

for name, d in [("Train", train_dataset), ("Validation", val_dataset), ("Test", test_dataset)]:
    print(f"{name:10s}: {len(d):5d} QA pairs from {len(set(d['context']))} articles")
print("\nTest question styles:")
print(pd.Series(test_dataset["style"]).value_counts().to_string())

## Step 5 — Load the base model (4-bit)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
print("Base model loaded.")

## Step 6 — Prompt format and training text

In [ ]:
alpaca_prompt = '''Below is a question about a law in Egypt (the Egyptian Civil Code, Law 131 of 1948), paired with an input that gives the text of the relevant article. Write an answer in formal Arabic that answers the question using only the article.

### Question:
{}

### Input:
{}

### Answer:
{}'''

EOS_TOKEN = tokenizer.eos_token

def to_text(ex):
    return {"text": alpaca_prompt.format(ex["question"], ex["context"], ex["answer"]) + EOS_TOKEN}

def add_text_and_filter(ds, name):
    ds = ds.map(to_text)
    lengths = [len(tokenizer(t)["input_ids"]) for t in ds["text"]]
    print(f"{name}: token length mean={np.mean(lengths):.0f} median={np.median(lengths):.0f} max={max(lengths)}; "
          f"{sum(l > MAX_SEQ_LENGTH for l in lengths)} longer than {MAX_SEQ_LENGTH}")
    return ds.add_column("n_tokens", lengths).filter(lambda ex: ex["n_tokens"] <= MAX_SEQ_LENGTH)

train_dataset = add_text_and_filter(train_dataset, "train")
val_dataset   = add_text_and_filter(val_dataset, "validation")
print("Training examples kept  :", len(train_dataset))
print("Validation examples kept:", len(val_dataset))

## Step 7 — Add the LoRA adapter and train

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

base_cfg = dict(
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=0.05,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
    report_to="none",
    dataset_text_field="text",
    dataset_num_proc=2,
    packing=False,
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

def build_trainer(response_only):
    last_err = None
    # Argument names differ between library versions, so try the known variants.
    for len_key in ("max_seq_length", "max_length"):
        for eval_key in ("eval_strategy", "evaluation_strategy"):
            try:
                args = SFTConfig(**{len_key: MAX_SEQ_LENGTH, eval_key: "epoch"}, **base_cfg)
            except TypeError as e:
                last_err = e
                continue
            tr = SFTTrainer(model=model, tokenizer=tokenizer,
                            train_dataset=train_dataset, eval_dataset=val_dataset, args=args)
            if response_only:
                tr = train_on_responses_only(tr, instruction_part="### Question:\n",
                                             response_part="### Answer:\n")
            return tr
    raise last_err

def trains_only_on_answers(tr):
    ex = tr.train_dataset[0]
    kept = [t for t in ex["labels"] if t != -100]
    if not kept:
        return False
    decoded = tokenizer.decode(kept, skip_special_tokens=True).strip()
    return decoded.startswith(train_dataset[0]["answer"].strip()[:30])

trainer = build_trainer(USE_RESPONSE_ONLY)

if USE_RESPONSE_ONLY:
    if trains_only_on_answers(trainer):
        print("Check passed: the loss is computed on the answer tokens only.")
    else:
        print("WARNING: answer-only masking did not work with this library version.")
        print("Falling back to training on the full text.")
        trainer = build_trainer(False)

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}, {gpu.total_memory / 1024**3:.1f} GB")

In [ ]:
trainer_stats = trainer.train()
print(f"\nTraining time : {trainer_stats.metrics['train_runtime'] / 60:.1f} min")
print(f"Train loss    : {trainer_stats.metrics.get('train_loss'):.4f}")
print(f"Best validation loss: {trainer.state.best_metric:.4f}  (best epoch kept)")

## Step 8 — Save the adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Keep the exact held-out test pairs with the adapter, so the RAG notebook evaluates only on unseen articles.
with open(f"{OUTPUT_DIR}/test_split.json", "w", encoding="utf-8") as f:
    json.dump(test_dataset.to_list(), f, ensure_ascii=False, indent=1)

zip_path = shutil.make_archive("/content/egyptian_lora_model", "zip", OUTPUT_DIR)
print("Saved adapter folder:", OUTPUT_DIR)
print("Zip:", zip_path)

In [ ]:
# Download the adapter now, before the Colab session ends.
# (Alternative: copy it to Google Drive with drive.mount + shutil.copy.)
from google.colab import files
files.download("/content/egyptian_lora_model.zip")

> **If the session restarts:** re-run Steps 1–4 and the prompt cell of Step 6, upload and unzip `egyptian_lora_model.zip`, and replace Step 5 with
> `model, tokenizer = FastLanguageModel.from_pretrained(model_name="/content/egyptian_lora_model", max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)`.
> The split is deterministic (`SPLIT_SEED`), so the test articles will be identical.

## Step 9 — Generate answers on the held-out articles

Greedy decoding (`do_sample=False`) so results are reproducible.

In [ ]:
from tqdm.auto import tqdm

FastLanguageModel.for_inference(model)

# Warn about test prompts that leave little room for the answer.
test_prompt_lens = [len(tokenizer(alpaca_prompt.format(e["question"], e["context"], ""))["input_ids"]) for e in test_dataset]
n_tight = sum(l > MAX_SEQ_LENGTH - 256 for l in test_prompt_lens)
print(f"Test prompts longer than {MAX_SEQ_LENGTH - 256} tokens: {n_tight}")

questions, contexts, references, predictions, styles = [], [], [], [], []

for ex in tqdm(test_dataset, desc="Generating"):
    prompt = alpaca_prompt.format(ex["question"], ex["context"], "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    pred = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    questions.append(ex["question"]); contexts.append(ex["context"])
    references.append(ex["answer"]);  predictions.append(pred)
    styles.append(ex["style"])

print("Generated answers:", len(predictions))

## Step 10 — Metrics (Arabic-aware)

Exact match, ROUGE and BLEU use the Arabic normalisation defined in Step 3.

In [ ]:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

class ArabicTokenizer:
    def tokenize(self, text):
        return normalize_ar(text).split()

# Sanity check: the default ROUGE tokenizer would score this 0.0
_chk = rouge_scorer.RougeScorer(["rouge1"], tokenizer=ArabicTokenizer()).score("يجب تنفيذ العقد بحسن نية", "يجب تنفيذ العقد")
print("ROUGE tokenizer sanity check (should be > 0):", round(_chk["rouge1"].fmeasure, 3))

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=ArabicTokenizer())
smooth = SmoothingFunction().method4

exact, r1, r2, rl, bleu = [], [], [], [], []
for pred, ref in zip(predictions, references):
    exact.append(normalize_ar(pred) == normalize_ar(ref))
    s = scorer.score(ref, pred)
    r1.append(s["rouge1"].fmeasure); r2.append(s["rouge2"].fmeasure); rl.append(s["rougeL"].fmeasure)
    pt, rt = normalize_ar(pred).split(), normalize_ar(ref).split()
    bleu.append(sentence_bleu([rt], pt, smoothing_function=smooth) if pt and rt else 0.0)

print("Done: exact match, ROUGE, BLEU.")

### Multilingual semantic similarity

In [ ]:
from sentence_transformers import SentenceTransformer, util

semantic_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

ref_emb  = semantic_model.encode(references,  convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True)
pred_emb = semantic_model.encode(predictions, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True)

semantic = util.cos_sim(pred_emb, ref_emb).diagonal().cpu().numpy()
print("Mean semantic similarity:", round(float(semantic.mean()), 4))

## Step 11 — Results

In [ ]:
def article_no(ctx):
    m = re.match(r"\s*المادة\s*(\d+)", ctx)
    return int(m.group(1)) if m else None

# Extra faithfulness check on the model's answers: share of answer words found in the article.
answer_faithfulness = [faithfulness(p, c) for p, c in zip(predictions, contexts)]

results_df = pd.DataFrame({
    "article": [article_no(c) for c in contexts],
    "style": styles,
    "question": questions,
    "context": contexts,
    "reference_answer": references,
    "generated_answer": predictions,
    "exact_match": exact,
    "rouge1": r1, "rouge2": r2, "rougeL": rl,
    "bleu": bleu,
    "semantic_similarity": semantic,
    "answer_in_article": answer_faithfulness,
})
results_df["semantic_match"] = results_df["semantic_similarity"] >= SEMANTIC_THRESHOLD

metric_cols = ["exact_match", "rouge1", "rouge2", "rougeL", "bleu",
               "semantic_similarity", "semantic_match", "answer_in_article"]

overall = results_df[metric_cols].mean()
by_style = results_df.groupby("style")[metric_cols].mean()
by_style["n"] = results_df.groupby("style").size()

print("=" * 75)
print("FINE-TUNED LLAMA-3.1-8B — EGYPTIAN CIVIL CODE QA (held-out articles)")
print("=" * 75)
print(f"Training QA pairs     : {len(train_dataset)}")
print(f"Held-out test pairs   : {len(results_df)}  ({results_df['article'].nunique()} articles)")
print(f"Exact Match           : {overall['exact_match'] * 100:.2f}%")
print(f"ROUGE-1 / 2 / L       : {overall['rouge1']:.4f} / {overall['rouge2']:.4f} / {overall['rougeL']:.4f}")
print(f"BLEU                  : {overall['bleu']:.4f}")
print(f"Mean semantic sim.    : {overall['semantic_similarity']:.4f}")
print(f"Semantic match @{SEMANTIC_THRESHOLD}  : {overall['semantic_match'] * 100:.2f}%")
print(f"Answer words in article: {overall['answer_in_article'] * 100:.1f}%")
print("\nBy question style:")
print(by_style.round(4).to_string())
print("=" * 75)

In [ ]:
csv_path = "/content/egyptian_law_llama_evaluation.csv"
results_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("Saved:", csv_path)

In [ ]:
# Look at real examples: 5 formal, 5 colloquial.
for style in ["formal", "colloquial"]:
    sub = results_df[results_df["style"] == style].head(5)
    for i, (_, row) in enumerate(sub.iterrows(), 1):
        print("\n" + "=" * 75)
        print(f"{style.upper()} EXAMPLE {i}   (article {row['article']})")
        print("=" * 75)
        print("QUESTION :", row["question"])
        print("REFERENCE:", row["reference_answer"])
        print("MODEL    :", row["generated_answer"])
        print(f"semantic={row['semantic_similarity']:.3f}  rougeL={row['rougeL']:.3f}")

In [ ]:
# The 10 weakest answers by semantic similarity: the best place to look for real legal errors.
worst = results_df.sort_values("semantic_similarity").head(10)
for _, row in worst.iterrows():
    print("\n" + "-" * 75)
    print(f"article {row['article']} | {row['style']} | semantic={row['semantic_similarity']:.3f}")
    print("QUESTION :", row["question"])
    print("REFERENCE:", row["reference_answer"])
    print("MODEL    :", row["generated_answer"])

In [ ]:
from google.colab import files
files.download("/content/egyptian_law_llama_evaluation.csv")